In [0]:
%python
CAT = "dev_insurance"

RULES = [
  # rule_id, source, rule_name, expression (must be TRUE to pass), severity
  ("PB01","premium_bordereau","line_id_not_null",      "bordereau_line_id IS NOT NULL", "error"),
  ("PB02","premium_bordereau","policy_number_format",  "policy_number RLIKE '^POL[0-9]{8}$'", "error"),
  ("PB03","premium_bordereau","broker_code_format",    "broker_code RLIKE '^[0-9]{5}$'", "error"),
  ("PB04","premium_bordereau","premium_is_decimal",    "try_cast(written_premium AS DECIMAL(18,2)) IS NOT NULL", "error"),
  ("PB05","premium_bordereau","date_is_valid",         "try_cast(transaction_date AS DATE) IS NOT NULL", "error"),
  ("PB06","premium_bordereau","ipt_commission_decimal","try_cast(ipt AS DECIMAL(18,2)) IS NOT NULL AND try_cast(commission AS DECIMAL(18,2)) IS NOT NULL", "error"),
  ("PB07","premium_bordereau","transaction_type_valid","transaction_type IN ('NEW_BUSINESS','ENDORSEMENT','CANCELLATION')", "error"),
  ("PB08","premium_bordereau","ipt_is_12_percent",     "abs(try_cast(ipt AS DECIMAL(18,2)) - try_cast(written_premium AS DECIMAL(18,2)) * 0.12) <= 0.01", "warning"),
  ("PB09","premium_bordereau","cancellation_is_negative","transaction_type <> 'CANCELLATION' OR try_cast(written_premium AS DECIMAL(18,2)) <= 0", "warning"),

  ("CE01","claim_events","event_id_not_null",  "event_id IS NOT NULL", "error"),
  ("CE02","claim_events","claim_number_format","claim_number RLIKE '^CLM[0-9]{7}$'", "error"),
  ("CE03","claim_events","event_type_valid",   "event_type IN ('REPORTED','ASSESSED','RESERVE_CHANGE','PAID','CLOSED')", "error"),
  ("CE04","claim_events","event_ts_valid",     "try_cast(event_ts AS TIMESTAMP) IS NOT NULL", "error"),
  ("CE05","claim_events","amount_is_decimal",  "try_cast(amount AS DECIMAL(18,2)) IS NOT NULL", "error"),
  ("CE06","claim_events","amount_not_negative","try_cast(amount AS DECIMAL(18,2)) >= 0", "warning"),

  ("PO01","policy","policy_number_format","policy_number RLIKE '^POL[0-9]{8}$'", "error"),
  ("PO02","policy","premium_is_decimal",  "try_cast(annual_premium AS DECIMAL(18,2)) IS NOT NULL", "error"),
  ("PO03","policy","dates_valid",         "try_cast(start_date AS DATE) IS NOT NULL AND try_cast(end_date AS DATE) IS NOT NULL", "error"),
  ("PO04","policy","updated_at_valid",    "try_cast(updated_at AS TIMESTAMP) IS NOT NULL", "error"),
  ("PO05","policy","status_valid",        "status IN ('ACTIVE','CANCELLED')", "error"),
  ("PO06","policy","end_after_start",     "try_cast(end_date AS DATE) > try_cast(start_date AS DATE)", "warning"),

  ("PH01","policyholder","id_format",       "policyholder_id RLIKE '^PH[0-9]{7}$'", "error"),
  ("PH02","policyholder","updated_at_valid","try_cast(updated_at AS TIMESTAMP) IS NOT NULL", "error"),
  ("PH03","policyholder","dob_valid_if_present", "date_of_birth IS NULL OR date_of_birth = '' OR try_cast(date_of_birth AS DATE) IS NOT NULL", "error"),
  ("PH04","policyholder","dob_present",     "date_of_birth IS NOT NULL AND date_of_birth <> ''", "warning"),
]

from pyspark.sql import functions as F
(spark.createDataFrame(RULES, "rule_id STRING, source STRING, rule_name STRING, expression STRING, severity STRING")
      .withColumn("active", F.lit(True))
      .write.mode("overwrite").saveAsTable(f"{CAT}.ops.dq_rules"))

display(spark.table(f"{CAT}.ops.dq_rules"))